## Agentic RAG with MCP (LangChain 버전)

원본 노트북의 OpenAI Agents SDK(`Agent` / `Runner` / `trace`) 구현을
LangChain `create_agent` + `langchain-mcp-adapters` 로 옮긴 버전입니다.

- 에이전트: `create_agent(model, tools, system_prompt=...)`
- 실행: `await agent.ainvoke({"messages": [...]})`
- trace 블록은 사용하지 않음

필요 패키지: `pip install langchain langchain-openai langchain-mcp-adapters python-dotenv`

### 윈도우 사용자 참고

윈도우 Jupyter 커널에서 stdio 방식 MCP 서버를 띄우면, 서버가 기록하는 stderr 스트림에
실제 파일 디스크립터가 없어 `io.UnsupportedOperation: fileno` 로 실패합니다.
아래 셀에서 stderr를 널 디바이스로 돌려 이 문제를 피합니다. 맥/리눅스에는 영향이 없습니다.

In [1]:
import os, functools, subprocess
from pathlib import Path
from dotenv import load_dotenv
from IPython.display import Markdown, display
from langchain.agents import create_agent                      # LangChain v1 에이전트 팩토리
from langchain_mcp_adapters.client import MultiServerMCPClient # 여러 MCP 서버를 한 클라이언트로 관리
from langchain_mcp_adapters.tools import load_mcp_tools        # MCP 세션 -> LangChain 툴 변환

load_dotenv(override=True)

# 윈도우 Jupyter 커널에서는 stdio MCP 서버의 stderr에 실제 파일 디스크립터가 없어
# io.UnsupportedOperation: fileno 로 실패하므로, stderr를 널 디바이스로 보내도록 감싼다.
import langchain_mcp_adapters.sessions as mcp_sessions
mcp_sessions.stdio_client = functools.partial(mcp_sessions.stdio_client, errlog=subprocess.DEVNULL)

### MCP 서버 두 개 설정

- Tavily: 웹 검색. 검색 외에 extract, crawl, map 등도 노출하므로 뒤에서 검색 툴만 골라 쓴다.
- Qdrant: 벡터 저장소. `QDRANT_LOCAL_PATH` 로 별도 DB 서버 없이 디스크에 저장하고,
  로컬 임베딩 모델을 쓰므로 추가 API 키가 필요 없다. 첫 저장/검색 때 모델을 내려받느라 한 번 멈춘다.

https://github.com/qdrant/mcp-server-qdrant

In [2]:
client = MultiServerMCPClient({
    # Tavily 웹 검색 서버 (npx로 실행)
    "tavily": {
        "transport": "stdio",
        "command": "npx",
        "args": ["-y", "tavily-mcp@latest"],
        # env를 지정하면 그 값만 자식 프로세스에 전달되므로, PATH 등을 잃지 않도록 os.environ을 합친다.
        "env": {**os.environ, "TAVILY_API_KEY": os.getenv("TAVILY_API_KEY")},
    },
    # Qdrant 벡터 저장소 서버 (uvx로 실행, 로컬 파일 모드)
    "qdrant": {
        "transport": "stdio",
        "command": "uvx",
        "args": ["mcp-server-qdrant"],
        "env": {**os.environ,
                "QDRANT_LOCAL_PATH": str(Path("memory/qdrant")),  # 별도 DB 서버 없이 디스크에 저장
                "COLLECTION_NAME": "knowledge"},                  # 사용할 컬렉션 이름
    },
})

In [ ]:
# 각 서버가 실제로 어떤 툴 이름을 노출하는지 확인 (툴 필터링 기준을 잡기 위한 용도)
async with client.session("tavily") as s_search, client.session("qdrant") as s_vector:
    print([t.name for t in await load_mcp_tools(s_search)])
    print([t.name for t in await load_mcp_tools(s_vector)])

## 조사 후 지식 베이스에 저장

RAG는 모델에 관련 문서를 붙여 답변의 근거를 만드는 방식이고, 보통은 벡터 저장소를 미리 채워둡니다.
Agentic RAG는 그 순서를 뒤집어, 에이전트가 무엇을 남길지 스스로 정하고 작업 중에 저장합니다.

아래 에이전트에는 Tavily 검색 툴과 Qdrant 저장/조회 툴을 함께 줍니다.

In [3]:
INSTRUCTIONS = """당신은 웹에서 주제를 조사하고, 나중에 활용할 지식 베이스를 구축하는 리서처입니다.
보관할 가치가 있는 내용을 알게 되면 반드시 지식 베이스에 저장하세요.
알고 있는 것이 무엇인지 질문받으면 지식 베이스를 검색해서 그 내용을 근거로 답변하세요."""

MODEL = "openai:gpt-5-mini"  # init_chat_model 형식의 모델 문자열

# 두 MCP 서버 세션을 동시에 열어두고, 세션이 살아 있는 동안 에이전트를 실행한다.
async with client.session("tavily") as s_search, client.session("qdrant") as s_vector:
    # tavily 서버는 search/extract/crawl 등 여러 툴을 노출하므로 검색 툴만 남기고 걸러낸다.
    tools = [t for t in await load_mcp_tools(s_search) if t.name == "tavily_search"]
    tools += await load_mcp_tools(s_vector)  # 저장/조회 툴 추가

    agent = create_agent(MODEL, tools, system_prompt=INSTRUCTIONS)
    result = await agent.ainvoke(
        {"messages": [{"role": "user",
                       "content": "엔비디아(Nvidia)에 대한 최신 뉴스를 조사하고, 핵심 사실들을 지식 베이스에 저장하세요."}]},
        config={"recursion_limit": 40},  # 도구 호출/추론 반복 상한
    )

display(Markdown(result["messages"][-1].content))  # 마지막 메시지가 최종 응답

완료했습니다. 최근 엔비디아 관련 주요 뉴스들을 조사해 핵심 사실들을 지식 베이스(Qdrant)에 저장해두었습니다.

저장한 요약(핵심 포인트)
1. 인수: 엔비디아가 Hugging Face를 약 129–130억 달러(약 $12.9–13.0B)에 인수 합의. 직원 유지용 지분 보상 프로그램으로 최대 약 $1B 포함. (소스: NVIDIA 블로그, CNBC, Bloomberg, Reuters 등)
2. 실적: 최신 분기(회계상 FY2027 Q2 보도 기준)에서 매출 약 $96.2B(전년대비 약 +106%), 데이터센터 매출 약 $89B(+117%)로 전체 매출의 대다수 차지. GAAP 순이익·EPS 대폭 증가(예: 순이익 약 $59.7B, EPS $2.46 등 보도). 경영진은 강력한 향후 성장 가이던스 제시(예: 차기 분기·향후 회계연도 고성장 전망). (소스: Finbold, Yahoo Finance, CNBC 등)
3. 제품·인프라: Blackwell 아키텍처의 급속 램프업; 에이전트용 CPU인 Vera(및 Vera Rubin 플랫폼) 대규모 출하 시작; NVLink Fusion·NVHBM 등 고대역폭 메모리·인프라 기술 발표·확장. (소스: NVIDIA 발표·보도자료)
4. 전략적 포지셔닝: GPU 공급사에서 전체 스택(플랫폼)으로 확장 — Hugging Face, Grok, Mellanox 등 인수 및 MediaTek 등 투자로 AI 생태계 확장 중. (소스: Yahoo Finance, Bloomberg 등)
5. 자본 환원: 분기 중 대규모 자사주 매입·배당 등으로 약 $26B 환원 보고, 남은 환매 승인 규모 약 $99B 등. (소스: Finbold, Yahoo)
6. 게임·소프트웨어: DLSS 5(신경 렌더링) 롤아웃 및 GeForce 드라이버(예: 616.64) 업데이트, GeForce NOW·Gamescom·IFA 관련 발표 등 게임 관련 소식 다수. (소스: NVIDIA 블로그, 유튜브 커뮤니티 보도)

저장 메타데이터
- 수집일: 2026-09-09
- 주요 출처: NVIDIA 공식 블로그, CNBC, Bloomberg, Reuters, Yahoo Finance, Finbold, Seeking Alpha, YouTube 등

원하시면
- 각 항목의 원문 링크(출처 기사들)를 제공해 드리거나,
- 최근 추가 뉴스(오늘·이번 주 기준)를 더 찾아 최신화해 드릴 수 있습니다.
어떻게 진행할까요?

## 저장된 지식만으로 답변

아래 에이전트에는 웹 검색 툴이 없고 지식 베이스만 있습니다.
엔비디아에 대해 말하는 내용은 앞 단계에서 저장한 것을 되살린 것이며, 이것이 RAG의 검색(retrieval) 쪽입니다.

In [4]:
# 이번에는 벡터스토어 세션만 열어 웹 검색 없이 저장된 지식만 사용한다.
async with client.session("qdrant") as s_vector:
    agent = create_agent(MODEL, await load_mcp_tools(s_vector), system_prompt=INSTRUCTIONS)
    result = await agent.ainvoke(
        {"messages": [{"role": "user",
                       "content": "당신의 지식 베이스를 바탕으로, 엔비디아(Nvidia)에 대한 최신 소식은 무엇인가요?"}]}
    )

display(Markdown(result["messages"][-1].content))

다음은 제 지식 베이스(최종 수집일: 2026-09-09)를 기준으로 한 엔비디아(NVIDIA) 최신 요약입니다.

핵심 요지
- 인수합병
  - Hugging Face 인수 합의: 약 129–130억 달러 규모(직원 인센티브 포함 최대 약 10억 달러 규모의 지분 기반 보상).(소스: NVIDIA 블로그, CNBC, Bloomberg 등)

- 재무실적(최근 분기 기준)
  - 기록적인 분기 실적 보고: 총매출 약 962억 달러(전년 대비 +106%대), Data Center 매출 약 890억 달러(전년 대비 +117%대)로 전체 매출의 대부분을 차지.
  - GAAP 순이익·주당순이익 대폭 증가(예: GAAP 순이익 약 597억 달러, EPS 약 $2.46 — 출처 보고치표에 따라 근사치).
  - 강한 향후 성장 가이던스(다음 분기 및 연간 가이던스가 높은 증가 전망을 제시).(소스: CNBC, Yahoo Finance, Finbold 등)

- 제품·인프라·기술
  - Blackwell AI 아키텍처의 빠른 전개(회사 역사상 가장 빠른 확장).
  - Vera CPU(에이전트 워크로드용 NVIDIA의 첫 CPU) 대규모 출하 시작.
  - Vera Rubin 플랫폼, NVLink Fusion, NVHBM(맞춤 HBM) 등 데이터센터/AI 인프라 기술 진전.

- 전략적 포지셔닝
  - 단순 GPU 판매사를 넘어서 전체 AI 스택(하드웨어·소프트웨어·모델·플랫폼) 기업으로 확장 중 — Hugging Face 등 인수와 생태계 투자로 이를 강화.

- 자본 정책
  - 대규모 자사주 환매·배당 등 주주환원 실행(최근 분기에 약 260억 달러 환원 보고, 잔여 환매 권한 표기 약 990억 달러 수준 — 보도별 근사치).

- 게임·소프트웨어
  - DLSS 5(신경 렌더링) 롤아웃 및 그래픽 드라이버(예: GeForce 드라이버 업데이트)·GeForce NOW 관련 소식과 게임행사 발표 등.

출처 및 주의사항
- 요약은 2026-09-03 ~ 2026-09-09 사이의 복수 보도(엔비디아 공식 블로그, CNBC, Bloomberg, Reuters, Yahoo Finance, Finbold, Seeking Alpha 등)를 종합한 것입니다. 수치들은 보도별 표기에 따라 근사치로 표기되었으니, 정확한 회계 수치나 법적 문구가 필요하면 원문(공시/보도자료)을 확인해 드리겠습니다.

원하시면
- 각 항목별 원문 링크(기사/공식 블로그) 제공
- Hugging Face 인수 조건 상세 분석
- 최신 재무제표(정확한 회계 수치) 정리
- 기술(Blackwell/Vera 등)의 의미와 경쟁사 대비 영향 분석

어떤 추가 정보를 원하시나요?

## 원본 대비 변경점

| 기존 (OpenAI Agents SDK) | 변경 (LangChain) |
| --- | --- |
| `Agent(..., mcp_servers=[...])` | `create_agent(model, tools, system_prompt=...)` |
| `await Runner.run(agent, "...")` | `await agent.ainvoke({"messages": [...]})` |
| `with trace("..."):` | 제거 |
| `create_static_tool_filter(allowed_tool_names=[...])` | 리스트 컴프리헨션으로 툴 이름 필터링 |
| `max_turns=20` | `config={"recursion_limit": 40}` |
| `result.final_output` | `result["messages"][-1].content` |

### 확인이 필요한 지점

- 라이브러리 선택: 여기서는 안정 버전인 `langchain-mcp-adapters`를 썼습니다. 공식 문서 기준으로 MCP 지원은
  `langchain.mcp` 네임스페이스로 흡수되어 `MultiServerMCPClient`가 `MCPAdapter` 한 클래스로 합쳐졌고,
  이쪽은 `langchain[mcp]>=1.4.0`이 필요한 베타입니다(임포트 시 경고, API 변경 가능).
- `client.get_tools()` 대신 `client.session(...)`을 쓴 이유: `MultiServerMCPClient`는 기본이 stateless라
  툴 호출마다 새 세션을 만듭니다. Qdrant 로컬 파일 모드에서는 매 호출마다 `uvx` 프로세스가 재기동되어 느리고,
  로컬 스토리지 잠금 충돌 가능성도 있습니다. 다만 이 충돌이 실제로 발생하는지는 확인하지 않았습니다.
- 원본의 `client_session_timeout_seconds`(60초 / 120초)에 정확히 대응하는 인자 이름은 현재 버전 기준으로
  확실하지 않아 생략했습니다. 첫 실행에서 임베딩 모델 다운로드로 타임아웃이 나면 `session_kwargs`로
  `read_timeout_seconds`를 넘기는 방식을 문서에서 확인하고 추가하세요.
- `recursion_limit`은 `max_turns`와 1:1이 아닙니다. LangGraph는 노드 실행 횟수를 세므로
  (모델 호출 + 툴 실행) 2스텝이 대략 1턴에 해당합니다.
- 첫 셀의 `langchain_mcp_adapters.sessions.stdio_client` 패치는 해당 모듈이 mcp SDK의 `stdio_client`를
  그 이름으로 임포트하고 있어야 동작합니다. `AttributeError`가 나면 설치된 패키지에서 실제 위치를 확인하세요.